# Agentic Twitter/X Data Collection
## Notebook 02 — Collecte de Données (Twitter API v2)
## Contexte : Tweets Éducatifs Cameroun (3ème, Première, Terminale / GCE O/A-Level)

**Objectif** : Collecter des tweets et leurs réponses sur les matières scolaires camerounaises
(systèmes francophone et anglophone), en miroir du pipeline YouTube (`01_data_collection`).

**Systèmes scolaires** : Francophone (BEPC/Bac) + Anglophone (GCE O/A-Level)

**Pipeline :**
1. Configuration & imports
2. Cache et utilitaires
3. Couche API Twitter v2
4. Phase B — Recherche des tweets candidats
5. Phase C — Déduplication
6. Phase D — Enrichissement des métriques
7. Phase E — Filtrage qualité
8. Phase F — Collecte des réponses (replies)
9. Résumé & vérification du dataset final

> Seed : 42 — Reproductible (NFR-02)

## 1 — Imports globaux

Avant d'exécuter quoi que ce soit, on importe toutes les bibliothèques nécessaires au pipeline.

| Bibliothèque | Rôle dans ce notebook |
|---|---|
| `os` | Lire les variables d'environnement (clé API depuis `.env`) |
| `json` | Lire et écrire les fichiers de cache (checkpoints) |
| `time` | Gérer les pauses entre appels API pour respecter le rate limit |
| `logging` | Tracer l'exécution dans un fichier log horodaté |
| `csv` | Écrire les résultats ligne par ligne sans tout charger en mémoire |
| `Path` | Manipuler les chemins de fichiers de façon portable (Windows/Linux) |
| `Optional, List` | Annotations de type pour une lecture du code plus claire |
| `datetime` | Convertir les timestamps Unix en dates ISO lisibles |
| `pandas` | Manipuler les DataFrames pour la déduplication et le filtrage |
| `requests` | Effectuer les appels HTTP vers l'API Twitter v2 |
| `tqdm` | Afficher une barre de progression pendant les longues boucles |
| `dotenv` | Charger automatiquement `.env` sans exposer les clés dans le code |

> **Pourquoi centraliser les imports ici ?** Si une dépendance est manquante (ex. `tqdm` non installé), l'erreur apparaît immédiatement avant tout traitement, ce qui évite d'attendre plusieurs minutes de collecte pour découvrir le problème.

In [ ]:
import os
import re
import json
import time
import logging
import csv
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
from collections import Counter

import pandas as pd
import requests
from tqdm import tqdm
from dotenv import load_dotenv

print('✅ Imports OK')

## 2 — Configuration centrale

Toute la configuration du pipeline est centralisée ici. Un lecteur peut modifier le comportement de la collecte **sans toucher au code des fonctions** — il suffit de changer les valeurs dans cette cellule.

### 2.1 — Authentification
On utilise le **Bearer Token** (authentification applicative OAuth 2.0) plutôt que OAuth 1.0a car :
- La recherche de tweets publics ne nécessite pas d'agir au nom d'un utilisateur
- Le Bearer Token suffit pour tous les endpoints `GET` en lecture seule de l'API v2
- `unquote()` est appliqué pour décoder automatiquement les tokens URL-encodés (certains outils copient le token avec `%3D` au lieu de `=`)

### 2.2 — Requêtes de recherche
Les requêtes sont structurées par **thématique → niveau → langue** pour couvrir les deux systèmes scolaires camerounais :
- **Francophone** : BEPC (3ème), Probatoire (Première), Baccalauréat (Terminale)
- **Anglophone** : GCE O-Level (Form 4/5), GCE A-Level (Lower/Upper 6th)

Chaque requête contient un mot-clé géographique (`Cameroun`/`Cameroon`) pour cibler les contenus locaux et réduire le bruit international.

### 2.3 — Paramètres de collecte
| Paramètre | Valeur | Justification |
|---|---|---|
| `MAX_TWEETS_PAR_REQUETE` | 10 | Limite maximale imposée par le **Free Tier** de l'API v2 |
| `TARGET_REPLIES` | 5 000 | Objectif minimum de réponses pour un dataset NLP exploitable |
| `SEUIL_MIN_REPLIES` | 2 | Garder uniquement les tweets qui ont généré de la conversation |
| `SEUIL_MIN_LIKES` | 1 | Éliminer les tweets sans aucun engagement (probablement du spam) |

### 2.4 — Paramètres de robustesse
| Paramètre | Valeur | Justification |
|---|---|---|
| `DELAI_ENTRE_APPELS` | 3 s | Le Free Tier autorise **15 req/15 min** → pause minimale pour ne pas être bloqué |
| `MAX_TENTATIVES` | 3 | Réessayer en cas d'erreur réseau ou de rate limit transitoire |
| `DELAI_ENTRE_TENTATIVES` | 60 s | Laisser la fenêtre de rate limit se réinitialiser avant de réessayer |

### 2.5 — Fichiers produits
Chaque phase écrit dans un fichier distinct pour garantir la **traçabilité** :
- `twitter_tweets_raw.csv` → tweets bruts issus de la recherche (Phase B)
- `twitter_tweets_deduplicated.csv` → après suppression des doublons (Phase C)
- `twitter_tweets_selected.csv` → après filtrage qualité (Phase E)
- `twitter_replies_raw.csv` → réponses collectées (Phase F)
- `twitter_cache_requetes.json` / `twitter_cache_replies.json` → checkpoints de reprise

In [ ]:
# ─── Authentification depuis .env ────────────────────────────────────────
load_dotenv()
from urllib.parse import unquote
TWITTER_BEARER_TOKEN = unquote(os.getenv('TWITTER_BEARER_TOKEN', ''))
if not TWITTER_BEARER_TOKEN:
    raise ValueError('TWITTER_BEARER_TOKEN manquant — vérifiez votre fichier .env')

# ─── Endpoints Twitter API v2 ─────────────────────────────────────────────
TWITTER_API_BASE       = 'https://api.twitter.com/2'
ENDPOINT_SEARCH        = f'{TWITTER_API_BASE}/tweets/search/recent'
ENDPOINT_TWEET_LOOKUP  = f'{TWITTER_API_BASE}/tweets'
ENDPOINT_REPLIES       = f'{TWITTER_API_BASE}/tweets/search/recent'

HEADERS = {'Authorization': f'Bearer {TWITTER_BEARER_TOKEN}'}

# ─── Requêtes — Contexte éducatif Cameroun ───────────────────────────────
# Structure : { thematique: { level: { langue: [requetes] } } }
REQUETES_PAR_THEMATIQUE = {
    'mathematiques': {
        'troisieme': {
            'fr': [
                'maths 3ème BEPC Cameroun',
                '#BEPCCameroun mathématiques',
                'exercices maths troisième Cameroun',
                'algèbre collège Cameroun',
            ],
        },
        'premiere': {
            'fr': [
                'maths Première Cameroun Probatoire',
                '#ProbataireCameroun mathématiques',
                'suites numériques Première Cameroun',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun mathématiques',
                'maths Terminale Cameroun révision',
                'intégrales Terminale Cameroun',
                'nombres complexes bac Cameroun',
            ],
        },
        'form4_5': {
            'en': [
                '#GCECameroon O-Level mathematics',
                'maths Form 4 Cameroon revision',
                'O-Level maths Cameroon',
            ],
        },
        'lower6th': {
            'en': [
                'A-Level maths Lower 6 Cameroon',
                '#ALevelCameroon mathematics',
                'calculus Lower 6th Cameroon',
            ],
        },
        'upper6th': {
            'en': [
                '#GCECameroon A-Level maths revision',
                'A-Level maths past questions Cameroon',
                'Upper 6th mathematics Cameroon',
            ],
        },
    },
    'svt_sciences': {
        'troisieme': {
            'fr': [
                'SVT 3ème BEPC Cameroun',
                '#BEPCCameroun SVT biologie',
                'sciences vie terre collège Cameroun',
            ],
        },
        'premiere': {
            'fr': [
                'SVT Première Cameroun Probatoire',
                '#ProbataireCameroun SVT',
                'biologie cellulaire Première Cameroun',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun SVT révision',
                'SVT Terminale Cameroun baccalauréat',
                'ADN réplication Terminale Cameroun',
            ],
        },
        'form4_5': {
            'en': [
                '#GCECameroon biology O-Level',
                'biology Form 4 Cameroon revision',
                'chemistry O-Level Cameroon',
            ],
        },
        'upper6th': {
            'en': [
                '#GCECameroon A-Level biology',
                'biology Upper 6th Cameroon',
                'A-Level science past papers Cameroon',
            ],
        },
    },
    'physique_chimie': {
        'troisieme': {
            'fr': [
                'physique chimie BEPC Cameroun',
                'électricité 3ème Cameroun cours',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun physique révision',
                'physique Terminale Cameroun baccalauréat',
            ],
        },
        'lower6th': {
            'en': [
                'chemistry A-Level Cameroon',
                '#ALevelCameroon physics',
            ],
        },
    },
    'francais_anglais': {
        'troisieme': {
            'fr': [
                'français BEPC Cameroun dissertation',
                '#BEPCCameroun français',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun français littérature',
                'baccalauréat français Cameroun révision',
            ],
        },
        'form4_5': {
            'en': [
                '#GCECameroon English O-Level',
                'English language Form 4 Cameroon',
            ],
        },
        'upper6th': {
            'en': [
                '#GCECameroon A-Level literature',
                'English literature Upper 6th Cameroon',
            ],
        },
    },
    'histoire_geo_eco': {
        'troisieme': {
            'fr': [
                'histoire géo BEPC Cameroun',
                '#BEPCCameroun histoire géographie',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun histoire géo économie',
                'économie Terminale Cameroun baccalauréat',
            ],
        },
        'upper6th': {
            'en': [
                '#GCECameroon A-Level history economics',
                'economics Upper 6th Cameroon',
            ],
        },
    },
}

# ─── Paramètres de collecte ───────────────────────────────────────────────
TARGET_REPLIES             = 5000  # Objectif minimum de réponses
MAX_TWEETS_PAR_REQUETE     = 10    # Max tweets par appel (free tier : max 10)
MAX_REPLIES_PAR_TWEET      = 100   # Max replies collectées par tweet

# ─── Paramètres de filtrage ───────────────────────────────────────────────
SEUIL_MIN_REPLIES          = 2     # Tweets avec moins de 2 réponses éliminés
SEUIL_MIN_LIKES            = 1     # Tweets avec 0 like éliminés

# ─── Paramètres de robustesse ─────────────────────────────────────────────
DELAI_ENTRE_APPELS         = 3     # Pause (sec) entre appels — rate limit : 15 req/15min
MAX_TENTATIVES             = 3
DELAI_ENTRE_TENTATIVES     = 60    # Pause (sec) avant retry (souvent rate limit)

# ─── Répertoires ──────────────────────────────────────────────────────────
REPERTOIRE_DATA     = Path('data')
REPERTOIRE_DATA_RAW = REPERTOIRE_DATA / 'raw'
REPERTOIRE_DATA.mkdir(parents=True, exist_ok=True)
REPERTOIRE_DATA_RAW.mkdir(parents=True, exist_ok=True)

FICHIER_TWEETS_BRUTS        = REPERTOIRE_DATA_RAW / 'twitter_tweets_raw.csv'
FICHIER_TWEETS_DEDUPLIQUES  = REPERTOIRE_DATA_RAW / 'twitter_tweets_deduplicated.csv'
FICHIER_TWEETS_SELECTIONNES = REPERTOIRE_DATA_RAW / 'twitter_tweets_selected.csv'
FICHIER_REPLIES_BRUTS       = REPERTOIRE_DATA_RAW / 'twitter_replies_raw.csv'
FICHIER_CACHE_REQUETES      = REPERTOIRE_DATA_RAW / 'twitter_cache_requetes.json'
FICHIER_CACHE_REPLIES       = REPERTOIRE_DATA_RAW / 'twitter_cache_replies.json'

total_requetes = sum(
    len(requetes)
    for niveaux in REQUETES_PAR_THEMATIQUE.values()
    for langues in niveaux.values()
    for requetes in langues.values()
)
print('\n✅ Configuration chargée — Contexte : Éducation Cameroun (Twitter/X)')
print(f'   Thématiques    : {list(REQUETES_PAR_THEMATIQUE.keys())}')
print(f'   Requêtes total : {total_requetes}')
print(f'   Objectif replies : {TARGET_REPLIES}')
print(f'   Répertoire     : {REPERTOIRE_DATA_RAW}')


## 3 — Système de cache

Le cache est le mécanisme de **reprise sur interruption** : si la collecte est coupée (coupure réseau, rate limit, arrêt manuel), on reprend exactement là où on s'est arrêté sans gaspiller de quota API.

### Pourquoi deux caches distincts ?
- **Cache requêtes** (`twitter_cache_requetes.json`) : enregistre quelles requêtes de recherche ont déjà été exécutées en Phase B. Clé = `thematique|level|langue|texte_de_la_requete`.
- **Cache replies** (`twitter_cache_replies.json`) : enregistre quels tweets ont déjà eu leurs réponses collectées en Phase F, ainsi que le total de réponses accumulées pour savoir si l'objectif `TARGET_REPLIES` est atteint.

### Pourquoi stocker en JSON ?
- Format **lisible** directement dans un éditeur pour inspecter l'avancement
- **Réinitialisable** facilement : supprimer le fichier = repartir de zéro pour cette phase
- Léger : les clés sont de courtes chaînes de caractères

> **Pour repartir de zéro** sur une phase : supprimer le fichier `.json` correspondant dans `data/raw/`. La prochaine exécution recréera le cache vide.

In [ ]:
# ─── Cache requêtes ──────────────────────────────────────────────────────
def charger_cache_requetes() -> dict:
    if FICHIER_CACHE_REQUETES.exists():
        with open(FICHIER_CACHE_REQUETES, encoding='utf-8') as f:
            return json.load(f)
    return {}

def sauvegarder_cache_requetes(cache: dict):
    with open(FICHIER_CACHE_REQUETES, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

def requete_deja_executee(thematique: str, level: str, langue: str, requete: str) -> bool:
    cache = charger_cache_requetes()
    return cache.get(f'{thematique}|{level}|{langue}|{requete}', False)

def marquer_requete_executee(thematique: str, level: str, langue: str, requete: str):
    cache = charger_cache_requetes()
    cache[f'{thematique}|{level}|{langue}|{requete}'] = True
    sauvegarder_cache_requetes(cache)

# ─── Cache replies ────────────────────────────────────────────────────────
def charger_cache_replies() -> dict:
    if FICHIER_CACHE_REPLIES.exists():
        with open(FICHIER_CACHE_REPLIES, encoding='utf-8') as f:
            return json.load(f)
    return {'processed_ids': [], 'total_replies': 0}

def sauvegarder_cache_replies(cache: dict):
    with open(FICHIER_CACHE_REPLIES, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

print('✅ Système de cache OK')


## 4 — Logger & utilitaires

### Pourquoi `logging` plutôt que `print` ?
- Chaque message est **horodaté automatiquement** — utile pour mesurer la durée réelle de chaque appel API et détecter les ralentissements
- Les messages sont écrits **simultanément dans un fichier** (`logs/twitter_collection.log`) et affichés à l'écran, ce qui permet de relire l'historique complet après une longue collecte
- Le niveau `INFO` filtre les messages de debug sans les supprimer (passer à `DEBUG` pour diagnostiquer un problème précis)

### Utilitaires métier
- **`construire_url_tweet(tweet_id, author_name)`** : reconstruit l'URL cliquable vers le tweet original sur `x.com`. L'`author_name` est inclus quand disponible pour une URL canonique (`x.com/user/status/id`), sinon on utilise l'URL générique (`x.com/i/web/status/id`).
- **`detecter_system(langue)`** : traduit le code langue (`fr`/`en`) en système scolaire (`francophone`/`anglophone`). Cette colonne permettra dans les notebooks suivants de segmenter les analyses par système sans devoir recroiser les données.

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s — %(levelname)s — %(message)s',
    handlers=[
        logging.FileHandler('logs/twitter_collection.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('twitter_collector')
Path('logs').mkdir(exist_ok=True)

def construire_url_tweet(tweet_id: str, author_name: str = '') -> str:
    if author_name:
        return f'https://x.com/{author_name}/status/{tweet_id}'
    return f'https://x.com/i/web/status/{tweet_id}'

def detecter_system(langue: str) -> str:
    return 'francophone' if langue == 'fr' else 'anglophone'

print('✅ Logger & utilitaires OK')


## 5 — Couche API Twitter v2

Cette section définit les trois fonctions d'accès à l'API. Centraliser les appels HTTP dans des fonctions dédiées permet de gérer les erreurs **une seule fois** plutôt que de répéter la logique dans chaque phase.

### `executer_requete_api()` — Wrapper HTTP avec retry
Traite tous les codes de réponse possibles de l'API v2 :
- **200 OK** : retourne le JSON de la réponse
- **429 Rate Limit** : lit l'en-tête `x-rate-limit-reset` pour savoir **exactement** quand la fenêtre se réinitialise, plutôt qu'attendre une durée fixe arbitraire
- **401 Unauthorized** : lève une exception immédiate — inutile de réessayer, le token est invalide ou expiré
- **Autres erreurs HTTP (5xx, etc.)** : réessaie jusqu'à `MAX_TENTATIVES` fois
- **Erreur réseau** (`ConnectionError`, timeout) : réessaie avec pause — couvre les coupures réseau transitoires

### `api_rechercher_tweets()` — Recherche de tweets candidats
- Ajoute `-is:retweet` pour **éliminer les retweets** : ils ne génèrent pas de conversation originale et dupliqueraient le contenu
- Ajoute `lang:{langue}` pour cibler la langue exacte et éviter les mélanges fr/en
- Demande `public_metrics` (likes, replies, retweets) en un seul appel : nécessaires pour le filtrage qualité en Phase E
- Utilise `expansions=author_id` + `user.fields=username` pour récupérer le nom d'auteur **dans le même appel** (sinon il faudrait un appel supplémentaire par tweet)

### `api_replies_tweet()` — Collecte des réponses d'un fil
- Recherche `conversation_id:{id} is:reply` : cible **toutes les réponses** du fil, pas seulement les réponses directes au tweet racine — les échanges imbriqués sont souvent les plus riches en contenu éducatif
- **Pagination via `next_token`** : l'API v2 renvoie au maximum 100 résultats par page ; la boucle suit les pages suivantes jusqu'à atteindre `max_results` ou épuiser les résultats disponibles
- Pause de `DELAI_ENTRE_APPELS` secondes entre chaque page pour respecter le rate limit même sur une conversation très longue

In [ ]:
def executer_requete_api(url: str, params: dict) -> Optional[dict]:
    """Wrapper HTTP avec retry et gestion du rate limit."""
    for tentative in range(1, MAX_TENTATIVES + 1):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                reset = int(resp.headers.get('x-rate-limit-reset', time.time() + DELAI_ENTRE_TENTATIVES))
                attente = max(reset - int(time.time()), DELAI_ENTRE_TENTATIVES)
                logger.warning(f'Rate limit atteint — attente {attente}s')
                time.sleep(attente)
            elif resp.status_code == 401:
                raise ValueError('Bearer Token invalide ou expiré — vérifiez .env')
            else:
                logger.warning(f'HTTP {resp.status_code} (tentative {tentative}/{MAX_TENTATIVES})')
                time.sleep(DELAI_ENTRE_TENTATIVES)
        except requests.RequestException as e:
            logger.error(f'Erreur réseau : {e} (tentative {tentative}/{MAX_TENTATIVES})')
            time.sleep(DELAI_ENTRE_TENTATIVES)
    return None


def api_rechercher_tweets(requete: str, langue: str, max_results: int = 10) -> Optional[dict]:
    """Recherche des tweets récents via search/recent."""
    # Exclure retweets et ajouter filtre langue
    query = f'{requete} -is:retweet lang:{langue}'
    params = {
        'query': query,
        'max_results': max_results,
        'tweet.fields': 'id,text,author_id,created_at,public_metrics,conversation_id,lang',
        'expansions': 'author_id',
        'user.fields': 'id,name,username',
    }
    return executer_requete_api(ENDPOINT_SEARCH, params)


def api_replies_tweet(conversation_id: str, max_results: int = 100) -> List[dict]:
    """Collecte toutes les réponses d'une conversation."""
    replies = []
    next_token = None

    while len(replies) < max_results:
        params = {
            'query': f'conversation_id:{conversation_id} is:reply',
            'max_results': min(100, max_results - len(replies)),
            'tweet.fields': 'id,text,author_id,created_at,public_metrics,in_reply_to_user_id,lang',
            'expansions': 'author_id',
            'user.fields': 'id,name,username',
        }
        if next_token:
            params['next_token'] = next_token

        reponse = executer_requete_api(ENDPOINT_REPLIES, params)
        if not reponse or not reponse.get('data'):
            break

        # Construire mapping author_id → username
        users = {u['id']: u.get('username', '') for u in reponse.get('includes', {}).get('users', [])}

        for tweet in reponse['data']:
            metrics = tweet.get('public_metrics', {})
            replies.append({
                'reply_id':           tweet['id'],
                'parent_tweet_id':    conversation_id,
                'conversation_id':    conversation_id,
                'text':               tweet.get('text', ''),
                'author_id':          tweet.get('author_id', ''),
                'author_name':        users.get(tweet.get('author_id', ''), ''),
                'created_at':         tweet.get('created_at', ''),
                'like_count':         metrics.get('like_count', 0),
                'reply_count':        metrics.get('reply_count', 0),
                'retweet_count':      metrics.get('retweet_count', 0),
                'lang':               tweet.get('lang', ''),
            })

        meta = reponse.get('meta', {})
        next_token = meta.get('next_token')
        if not next_token:
            break
        time.sleep(DELAI_ENTRE_APPELS)

    return replies


print('✅ Couche API Twitter v2 OK')


## 6 — Phase B : Recherche des tweets candidats

C'est la première phase de collecte réelle. L'objectif est d'obtenir une liste de tweets **pertinents et engageants** sur lesquels on collectera les réponses.

### Pourquoi chercher des tweets avant les réponses ?
L'API Twitter v2 ne permet pas de chercher directement des réponses par thématique. Il faut d'abord trouver les tweets racines qui ont généré des conversations, puis collecter leurs réponses via leur `conversation_id`.

### Logique de la boucle
1. **Vérification du cache** : si la requête a déjà été exécutée lors d'une session précédente, on la saute pour économiser le quota API
2. **Appel API** : `api_rechercher_tweets()` renvoie jusqu'à 10 tweets par requête (limite du Free Tier)
3. **Enrichissement** : on ajoute les métadonnées contextuelles (`thematique`, `level`, `system`, `langue`) absentes de la réponse API mais essentielles pour les analyses ultérieures
4. **Écriture incrémentale** : les résultats sont ajoutés au CSV immédiatement après chaque requête (mode append) — si le notebook est interrompu, les tweets déjà collectés ne sont pas perdus
5. **Marquage du cache** : la requête est enregistrée comme exécutée avant la pause, garantissant qu'un crash pendant le `sleep` ne cause pas de double appel

> **Note sur le `conversation_id`** : cet identifiant est égal au `tweet_id` du tweet racine du fil. Il sera utilisé en Phase F pour retrouver toutes les réponses appartenant au même fil, même imbriquées à plusieurs niveaux.

In [ ]:
COLONNES_TWEETS = [
    'tweet_id', 'text', 'author_id', 'author_name', 'created_at',
    'like_count', 'reply_count', 'retweet_count', 'impression_count',
    'conversation_id', 'lang', 'langue', 'thematique', 'level',
    'system', 'search_query', 'url'
]

def executer_phase_recherche() -> pd.DataFrame:
    """Recherche des tweets par thématique, niveau et langue."""
    logger.info('Phase B — Recherche des tweets candidats')

    toutes_requetes = [
        (thematique, level, langue, requete)
        for thematique, niveaux in REQUETES_PAR_THEMATIQUE.items()
        for level, langues in niveaux.items()
        for langue, requetes in langues.items()
        for requete in requetes
    ]
    logger.info(f'{len(toutes_requetes)} requêtes planifiées')
    nb_nouveaux = 0

    for thematique, level, langue, requete in tqdm(toutes_requetes, desc='Recherche tweets'):
        if requete_deja_executee(thematique, level, langue, requete):
            continue

        reponse = api_rechercher_tweets(requete, langue, MAX_TWEETS_PAR_REQUETE)
        marquer_requete_executee(thematique, level, langue, requete)

        if not reponse or not reponse.get('data'):
            time.sleep(DELAI_ENTRE_APPELS)
            continue

        users = {u['id']: u.get('username', '') for u in reponse.get('includes', {}).get('users', [])}

        lignes = []
        for tweet in reponse['data']:
            metrics = tweet.get('public_metrics', {})
            author_id = tweet.get('author_id', '')
            author_name = users.get(author_id, '')
            tweet_id = tweet['id']
            lignes.append({
                'tweet_id':         tweet_id,
                'text':             tweet.get('text', ''),
                'author_id':        author_id,
                'author_name':      author_name,
                'created_at':       tweet.get('created_at', ''),
                'like_count':       metrics.get('like_count', 0),
                'reply_count':      metrics.get('reply_count', 0),
                'retweet_count':    metrics.get('retweet_count', 0),
                'impression_count': metrics.get('impression_count', 0),
                'conversation_id':  tweet.get('conversation_id', tweet_id),
                'lang':             tweet.get('lang', ''),
                'langue':           langue,
                'thematique':       thematique,
                'level':            level,
                'system':           detecter_system(langue),
                'search_query':     requete,
                'url':              construire_url_tweet(tweet_id, author_name),
            })

        if lignes:
            df_batch = pd.DataFrame(lignes, columns=COLONNES_TWEETS)
            mode = 'a' if FICHIER_TWEETS_BRUTS.exists() else 'w'
            df_batch.to_csv(FICHIER_TWEETS_BRUTS, mode=mode,
                            header=(mode == 'w'), index=False, encoding='utf-8')
            nb_nouveaux += len(lignes)

        time.sleep(DELAI_ENTRE_APPELS)

    if FICHIER_TWEETS_BRUTS.exists():
        df = pd.read_csv(FICHIER_TWEETS_BRUTS, encoding='utf-8')
        logger.info(f'Phase B terminée : {len(df)} tweets candidats ({nb_nouveaux} nouveaux)')
        return df
    return pd.DataFrame(columns=COLONNES_TWEETS)


# EXÉCUTION
df_tweets_bruts = executer_phase_recherche()
print(f'\n📊 Tweets bruts : {len(df_tweets_bruts)}')
df_tweets_bruts.head()


## 7 — Phase C : Déduplication

La déduplication est indispensable car un même tweet peut apparaître dans plusieurs requêtes de recherche.

### Pourquoi des doublons apparaissent-ils ?
Un tweet comme *"Révision maths Terminale #BacCameroun"* sera renvoyé par la requête `maths Terminale Cameroun révision` ET par `#BacCameroun mathématiques` — deux requêtes différentes qui matchent le même tweet.

### Choix de déduplication
- **Clé : `tweet_id`** — identifiant unique garanti par Twitter, sans ambiguïté
- On garde la **première occurrence** (la plus ancienne dans le CSV) pour préserver la cohérence des métadonnées associées
- Le fichier brut `twitter_tweets_raw.csv` est conservé intact pour permettre de rejouer cette étape avec des critères différents si besoin

### Distribution affichée
Les tableaux de distribution par système/langue et par thématique/niveau permettent de détecter un **déséquilibre** dans les données avant de continuer. Si une thématique est sur-représentée, on peut ajuster les requêtes et relancer.

In [ ]:
logger.info('Phase C — Déduplication des tweets')

nb_avant = len(df_tweets_bruts)
df_deduplique = df_tweets_bruts.drop_duplicates(subset='tweet_id').reset_index(drop=True)
nb_apres = len(df_deduplique)

df_deduplique.to_csv(FICHIER_TWEETS_DEDUPLIQUES, index=False, encoding='utf-8')

logger.info(f'Phase C terminée : {nb_avant} → {nb_apres} tweets uniques')
print(f'\n✅ Déduplication : {nb_avant} → {nb_apres} tweets uniques')
print('\n─── Distribution par système ───')
print(df_deduplique.groupby(['system', 'langue']).size().reset_index(name='nb_tweets').to_string(index=False))
print('\n─── Distribution par thématique ───')
print(df_deduplique.groupby(['thematique', 'level']).size().reset_index(name='nb_tweets').to_string(index=False))


## 8 — Phase E : Filtrage qualité

On ne collecte pas les réponses de **tous** les tweets trouvés, seulement de ceux qui ont déjà prouvé leur capacité à générer de la conversation. Cela optimise l'utilisation du quota API limité.

### Règle F1 — Nombre minimum de réponses (`reply_count >= 2`)
Un tweet avec 0 ou 1 réponse ne produira pas de données d'entraînement utiles. On fixe le seuil à **2 réponses minimum** pour garantir au moins un échange réel. Ce seuil est volontairement bas pour ne pas trop réduire le corpus.

### Règle F2 — Engagement minimum (`like_count >= 1`)
Un tweet sans aucun like est suspect : il peut s'agir de spam, d'un compte automatisé ou d'un contenu complètement hors-sujet qui aurait passé les filtres de recherche. Au moins **1 like** confirme qu'un humain a jugé le tweet pertinent.

### Pourquoi ne pas filtrer par langue ici ?
Le filtre de langue est déjà appliqué **au moment de la requête API** (`lang:fr` ou `lang:en`). Refiltrer ici serait redondant. La colonne `lang` (détectée par Twitter) reste disponible pour des analyses ultérieures si nécessaire.

> **Résultat attendu** : une réduction significative du nombre de tweets, mais chacun des tweets restants a une forte probabilité de générer des réponses de qualité lors de la Phase F.

In [ ]:
logger.info('Phase E — Filtrage des tweets')

df_filtre = df_deduplique.copy()
for col in ['like_count', 'reply_count']:
    df_filtre[col] = pd.to_numeric(df_filtre[col], errors='coerce').fillna(0)

nb_initial = len(df_filtre)

# Règle F1 : replies minimum
df_filtre = df_filtre[df_filtre['reply_count'] >= SEUIL_MIN_REPLIES]
logger.info(f'F1 (reply_count >= {SEUIL_MIN_REPLIES}) : {nb_initial} → {len(df_filtre)}')

# Règle F2 : likes minimum
n2 = len(df_filtre)
df_filtre = df_filtre[df_filtre['like_count'] >= SEUIL_MIN_LIKES]
logger.info(f'F2 (like_count >= {SEUIL_MIN_LIKES}) : {n2} → {len(df_filtre)}')

df_filtre = df_filtre.reset_index(drop=True)
df_filtre.to_csv(FICHIER_TWEETS_SELECTIONNES, index=False, encoding='utf-8')

logger.info(f'Phase E terminée : {nb_initial} → {len(df_filtre)} tweets retenus')
print(f'\n✅ Filtrage : {nb_initial} → {len(df_filtre)} tweets retenus')
print('\n─── Tweets sélectionnés par thématique/niveau ───')
print(df_filtre.groupby(['thematique', 'level']).size().reset_index(name='nb_tweets').to_string(index=False))


## 9 — Phase F : Collecte des réponses (replies)

C'est la phase centrale du pipeline : on collecte les **réponses** (équivalent des commentaires YouTube) qui constituent le dataset final pour l'analyse NLP.

### Pourquoi les réponses et non les tweets eux-mêmes ?
Les réponses représentent les **interactions authentiques** entre apprenants : questions, corrections, explications, encouragements. Ce sont ces textes qui seront utilisés pour entraîner ou évaluer des modèles de traitement du langage.

### Logique de la boucle
1. **Reprise depuis le checkpoint** : les tweets déjà traités sont chargés depuis `twitter_cache_replies.json` — on reprend exactement là où on s'était arrêté
2. **Arrêt anticipé** : dès que `TARGET_REPLIES` (5 000) est atteint, on arrête sans traiter les tweets restants — économie de quota API
3. **Appel API** : `api_replies_tweet()` collecte jusqu'à `MAX_REPLIES_PAR_TWEET` réponses par tweet, avec pagination automatique
4. **Écriture immédiate** : chaque batch de réponses est écrit dans le CSV et le fichier est flushé — protection contre toute interruption
5. **Mise à jour du checkpoint** : le cache est sauvegardé après chaque tweet traité, pas seulement en fin de boucle

### Pourquoi `conversation_id` et non `tweet_id` pour la recherche ?
Sur Twitter, une réponse à une réponse appartient au même `conversation_id` que le tweet racine. En cherchant par `conversation_id`, on capture **tout le fil**, y compris les sous-discussions imbriquées — souvent les plus riches en contenu.

### Barre de progression
La barre `[████░░░░░░░░░░░░░░░░]` affiche l'avancement vers `TARGET_REPLIES`. Si la barre stagne, c'est généralement le signe que les tweets restants n'ont plus de nouvelles réponses à collecter.

In [ ]:
COLONNES_REPLIES = [
    'reply_id', 'parent_tweet_id', 'conversation_id', 'text',
    'author_id', 'author_name', 'created_at',
    'like_count', 'reply_count', 'retweet_count', 'lang'
]

# Reprise depuis checkpoint
cache_replies = charger_cache_replies()
processed_ids = set(cache_replies.get('processed_ids', []))
total_replies = cache_replies.get('total_replies', 0)

if processed_ids:
    print(f'🔄 Reprise checkpoint : {len(processed_ids)} tweets déjà traités, {total_replies} réponses collectées')

mode = 'a' if FICHIER_REPLIES_BRUTS.exists() and processed_ids else 'w'

with open(FICHIER_REPLIES_BRUTS, mode, newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=COLONNES_REPLIES, extrasaction='ignore')
    if mode == 'w':
        writer.writeheader()

    for i, row in enumerate(df_filtre.itertuples(), 1):
        if total_replies >= TARGET_REPLIES:
            print(f'\n🎯 Objectif atteint : {total_replies} réponses — arrêt anticipé.')
            break

        tweet_id = str(row.tweet_id)
        conv_id  = str(row.conversation_id)

        if tweet_id in processed_ids:
            continue

        remaining = TARGET_REPLIES - total_replies
        print(f'[{i}/{len(df_filtre)}] 💬 {str(row.text)[:60]} | {total_replies}/{TARGET_REPLIES} (+{remaining} manquants)')

        replies = api_replies_tweet(conv_id, MAX_REPLIES_PAR_TWEET)

        if replies:
            writer.writerows(replies)
            f.flush()
            total_replies += len(replies)
            pct = min(100, round(total_replies / TARGET_REPLIES * 100))
            bar = '█' * (pct // 5) + '░' * (20 - pct // 5)
            print(f'   ✅ +{len(replies)} réponses → {total_replies}/{TARGET_REPLIES} [{bar}] {pct}%')
        else:
            print(f'   ⛔ Aucune réponse trouvée')

        processed_ids.add(tweet_id)
        cache_replies['processed_ids'] = list(processed_ids)
        cache_replies['total_replies'] = total_replies
        cache_replies['updated'] = datetime.utcnow().isoformat()
        sauvegarder_cache_replies(cache_replies)

        time.sleep(DELAI_ENTRE_APPELS)

if total_replies >= TARGET_REPLIES:
    print(f'\n✅ Objectif atteint : {total_replies} réponses collectées')
else:
    print(f'\n⚠️  Collecte terminée — {total_replies}/{TARGET_REPLIES} réponses.')
    print(f'   → Relance possible : {len(processed_ids)} tweets déjà traités (checkpoint actif)')
    print(f'   → Pour collecter plus : élargir les mots-clés ou baisser SEUIL_MIN_REPLIES')


## 10 — Résumé & vérification du dataset final

Cette cellule synthétise les résultats de toutes les phases et vérifie que les fichiers attendus ont bien été produits.

### Ce que l'on vérifie
- **Funnel de collecte** : `tweets bruts → dédupliqués → sélectionnés → réponses` — chaque étape doit réduire le volume ; une augmentation serait un signal d'anomalie
- **Existence et taille des fichiers** : un fichier absent ou vide indique qu'une phase n'a pas produit de résultats (quota épuisé, requêtes trop restrictives)
- **Distribution des réponses par tweet** : la médiane et la moyenne révèlent si quelques tweets monopolisent les réponses (biais de représentation) ou si la collecte est bien distribuée

### Interprétation des statistiques de distribution
| Indicateur | Signal positif | Signal négatif |
|---|---|---|
| Médiane > 5 réponses/tweet | Distribution équilibrée | Médiane = 1 : tweets trop peu engagés |
| Max >> Médiane × 10 | Quelques tweets viraux (OK) | Max >> Médiane × 100 : biais fort |
| Total replies ≥ 5 000 | Objectif atteint | < 5 000 : relancer avec SEUIL_MIN_REPLIES = 1 |

> **Étape suivante** : le fichier `twitter_replies_raw.csv` alimente le notebook de preprocessing (détection de langue, nettoyage du texte, analyse de sentiment).

In [ ]:
print('=' * 60)
print('   RÉSUMÉ DE LA COLLECTE TWITTER/X')
print('=' * 60)
print(f'  Tweets candidats (bruts)       : {len(df_tweets_bruts)}')
print(f'  Tweets après déduplication     : {len(df_deduplique)}')
print(f'  Tweets sélectionnés (filtrés)  : {len(df_filtre)}')
print(f'  Réponses collectées            : {total_replies}')
print('=' * 60)

print('\n📁 Fichiers produits :')
for fichier in [
    FICHIER_TWEETS_BRUTS,
    FICHIER_TWEETS_DEDUPLIQUES,
    FICHIER_TWEETS_SELECTIONNES,
    FICHIER_REPLIES_BRUTS,
]:
    taille = fichier.stat().st_size / 1024 if fichier.exists() else 0
    statut = '✅' if fichier.exists() else '❌'
    print(f'  {statut} {fichier.name:<50} ({taille:.1f} Ko)')

# Distribution des réponses
if FICHIER_REPLIES_BRUTS.exists():
    df_replies = pd.read_csv(FICHIER_REPLIES_BRUTS, encoding='utf-8')
    print(f'\n─── Distribution réponses/tweet ───')
    stats = df_replies.groupby('parent_tweet_id')['reply_id'].count()
    print(f'  Min     : {stats.min()}')
    print(f'  Médiane : {stats.median():.1f}')
    print(f'  Max     : {stats.max()}')
    print(f'  Moyenne : {stats.mean():.1f}')

print('\n🔜 Prochaine étape : preprocessing & analyse de sentiment')
